<a href="https://colab.research.google.com/github/abhijadhav14/Data-Analytics-Using-Python/blob/main/classifica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from google.colab import drive
drive.mount('/content/drive')
spark = SparkSession.builder \
    .appName("IrisClassification") \
    .getOrCreate()

df = spark.read.csv(
    "/content/drive/MyDrive/data/Iris.csv",
    header=True,
    inferSchema=True
)
df.show()

# Convert species names to numeric labels
indexer = StringIndexer(
    inputCol="Species",
    outputCol="label"
)
df = indexer.fit(df).transform(df)
# Combine feature columns into a single vector
assembler = VectorAssembler(
    inputCols=[
        "SepalLengthCm",
        "SepalWidthCm",
        "PetalLengthCm",
        "PetalWidthCm"
    ],
    outputCol="features"
)
data = assembler.transform(df)
# Split data
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)
# Create classifier
dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label"
)

# Train model
model = dt.fit(train_data)
# Predict
predictions = model.transform(test_data)
# Display results
predictions.select(
    "Species",
    "label",
    "prediction"
).show()
# Evaluate accuracy
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy = evaluator.evaluate(predictions)
print("Accuracy =", accuracy)
spark.stop()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
+---+-------------+------------+-------------+------------+-----------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|
+---+-------------+------------+-------------+------------+-----------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|
|  6|          5.4|         3.9|          1.7|         0.4|Iris-setosa|
|  7|          4.6|         3.4|          1.4|         0.3|Iris-setosa|
|  8|          5.0|         3.4|          1.5|         0.2|Iris-setosa|
|  9|          4.4|         2.9|          1.4|         0.2|Iris-setosa|
| 10|  